# Drone Delivery Monitoring & Failure Analytics Pipeline


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import *

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DDMFA")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 08:59:03 WARN Utils: Your hostname, NOMAAN-ANV15, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 08:59:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/ddmfa/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nomaan/.ivy2.5.2/cache
The jars for the packages stored in: /home/nomaan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab216608-9431-4ee2-9977-9d49d28f4729;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central


# Bronze Layer: Raw Ingestion

**What this notebook does:**
- Reads all 3 raw CSVs from the Unity Catalog Volume
- Adds ingestion metadata (`ingestion_time`, `source_file`) to every record
- Deduplicates `flight_logs` on `log_id` with a logged audit count
- Writes 3 Bronze Delta tables to `ddmfa_catalog.drone_schema`

Bronze = take in raw data + add traceability metadata.

In [3]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import DataFrame

# Use: log pre/post deduplication counts
def dedup_audit(df_before: DataFrame, df_after: DataFrame, pk: str, table: str):
    pre  = df_before.count()
    post = df_after.count()
    dropped = pre - post
    print(f"[DEDUP AUDIT] {table}")
    print(f"  Before : {pre:,} rows")
    print(f"  After  : {post:,} rows")
    print(f"  Dropped: {dropped:,} duplicate rows on '{pk}'")
    return dropped

## 1. Ingest `drones.csv` → `bronze_drones`

In [4]:
# Read raw CSV
df_drones_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DRONES_FILE))
)

# Defensive dedup on primary key
df_drones_deduped = df_drones_raw.dropDuplicates(["drone_id"])
dedup_audit(df_drones_raw, df_drones_deduped, "drone_id", "bronze_drones")

# Add ingestion metadata
df_bronze_drones = (
    df_drones_deduped
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file",    lit("drones.csv"))
)

# Write to Delta table
(
    df_bronze_drones.write
    .format("delta")
    .mode("overwrite")
    .save(str(BRONZE_DIR / "bronze_drones"))
)

print(f"\n✅ bronze_drones written — {df_bronze_drones.count():,} rows")
df_bronze_drones.printSchema()

[DEDUP AUDIT] bronze_drones
  Before : 100 rows
  After  : 100 rows
  Dropped: 0 duplicate rows on 'drone_id'


26/07/30 09:01:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                


✅ bronze_drones written — 100 rows
root
 |-- drone_id: string (nullable = true)
 |-- model: string (nullable = true)
 |-- max_range_km: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



## 2. Ingest `deliveries.csv` → `bronze_deliveries`

In [5]:
# Read raw CSV
df_deliveries_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DELIVERIES_FILE))
)

# Defensive dedup on primary key
df_deliveries_deduped = df_deliveries_raw.dropDuplicates(["delivery_id"])
dedup_audit(
    df_deliveries_raw,
    df_deliveries_deduped,
    "delivery_id",
    "bronze_deliveries"
)

# Add ingestion metadata
df_bronze_deliveries = (
    df_deliveries_deduped
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", lit("deliveries.csv"))
)

# Write to Delta table
(
    df_bronze_deliveries.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(BRONZE_DIR / "bronze_deliveries"))
)

print(f"\n✅ bronze_deliveries written — {df_bronze_deliveries.count():,} rows")
df_bronze_deliveries.printSchema()

[DEDUP AUDIT] bronze_deliveries
  Before : 5,000 rows
  After  : 5,000 rows
  Dropped: 0 duplicate rows on 'delivery_id'



✅ bronze_deliveries written — 5,000 rows
root
 |-- delivery_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



## 3. Ingest `flight_logs.csv` → `bronze_flight_logs`
This table has **150 intentionally injected duplicate rows** — dedup audit is meaningful here.

In [6]:
# Read raw CSV
df_logs_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(FLIGHT_LOGS_FILE))
)

# Real dedup — 150 duplicates expected on log_id
df_logs_deduped = df_logs_raw.dropDuplicates(["log_id"])
dropped = dedup_audit(
    df_logs_raw,
    df_logs_deduped,
    "log_id",
    "bronze_flight_logs"
)

if dropped == 150:
    print("  ✅ Dedup count matches expectation (150)")
else:
    print(f"  ⚠️ Expected 150 dropped, got {dropped} — verify source file")

# Add ingestion metadata
df_bronze_logs = (
    df_logs_deduped
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", lit("flight_logs.csv"))
)

# Write to Delta table
(
    df_bronze_logs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(BRONZE_DIR / "bronze_flight_logs"))
)

print(f"\n✅ bronze_flight_logs written — {df_bronze_logs.count():,} rows")
df_bronze_logs.printSchema()

[DEDUP AUDIT] bronze_flight_logs
  Before : 30,150 rows
  After  : 30,000 rows
  Dropped: 150 duplicate rows on 'log_id'
  ✅ Dedup count matches expectation (150)



✅ bronze_flight_logs written — 30,000 rows
root
 |-- log_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- delivery_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- battery_level: double (nullable = true)
 |-- gps_signal: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- status: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



## 4. Validation — Row Counts, Samples, Delta Metadata

In [7]:
# ── Row count summary ─────────────────────────────────────────────────

tables = {
    "bronze_drones": 100,
    "bronze_deliveries": 5_000,
    "bronze_flight_logs": 30_000,
}

print("── Bronze Layer Row Count Validation ───────────────────")

all_pass = True

for table, expected in tables.items():
    actual = (
        spark.read
        .format("delta")
        .load(str(BRONZE_DIR / table))
        .count()
    )

    status = "✅" if actual == expected else "⚠️"

    if actual != expected:
        all_pass = False

    print(f"{status} {table:<25} {actual:>6,} (expected {expected:,})")

print()

if all_pass:
    print("✅ All counts match expected values.")
else:
    print("⚠️ One or more counts differ — review source files.")

── Bronze Layer Row Count Validation ───────────────────
✅ bronze_drones                100 (expected 100)
✅ bronze_deliveries          5,000 (expected 5,000)
✅ bronze_flight_logs        30,000 (expected 30,000)

✅ All counts match expected values.


In [8]:
# ── Sample records ────────────────────────────────────────────────────

print("── bronze_drones (3 rows) ──────────────────────────────")
(
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_drones"))
    .show(3, truncate=False)
)

print("── bronze_deliveries (3 rows) ──────────────────────────")
(
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_deliveries"))
    .show(3, truncate=False)
)

print("── bronze_flight_logs (3 rows) ─────────────────────────")
(
    spark.read
    .format("delta")
    .load(str(BRONZE_DIR / "bronze_flight_logs"))
    .show(3, truncate=False)
)

── bronze_drones (3 rows) ──────────────────────────────
+--------+----------+------------+--------------------------+-----------+
|drone_id|model     |max_range_km|ingestion_time            |source_file|
+--------+----------+------------+--------------------------+-----------+
|D001    |Wing-G2   |57.4        |2026-07-30 09:01:06.867758|drones.csv |
|D002    |Skydio-D2 |75.5        |2026-07-30 09:01:06.867758|drones.csv |
|D003    |Zipline-R1|98.0        |2026-07-30 09:01:06.867758|drones.csv |
+--------+----------+------------+--------------------------+-----------+
only showing top 3 rows
── bronze_deliveries (3 rows) ──────────────────────────
+-----------+--------+---------------+-----------+-----------+-------------------+-------------------+-------+--------------------------+--------------+
|delivery_id|drone_id|source         |destination|distance_km|start_time         |end_time           |status |ingestion_time            |source_file   |
+-----------+--------+---------------+

In [9]:
tables = [
    "bronze_drones",
    "bronze_deliveries",
    "bronze_flight_logs",
]

for table in tables:
    detail = (
        spark.read
        .format("delta")
        .load(str(BRONZE_DIR / table))
        .inputFiles()
    )

    print(table)
    print("  Format   : delta")
    print(f"  Location : {BRONZE_DIR / table}")
    print()

bronze_drones
  Format   : delta
  Location : /home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/bronze/bronze_drones

bronze_deliveries
  Format   : delta
  Location : /home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/bronze/bronze_deliveries

bronze_flight_logs
  Format   : delta
  Location : /home/nomaan/projects/Drone_Delivery_Monitoring_and_Failure_Analysis_Pipeline/bronze/bronze_flight_logs



## 5. Null Count (Reference for Silver)
Bronze does not fix nulls. This check documents their counts so Silver can validate its cleaning logic against a baseline.

In [10]:
from pyspark.sql.functions import col, sum as spark_sum

def null_audit(table_name: str):
    df = (
        spark.read
        .format("delta")
        .load(str(BRONZE_DIR / table_name))
    )

    total = df.count()

    print(f"── {table_name} ({total:,} rows) ─────────────────────────")

    null_counts = (
        df.select([
            spark_sum(col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ])
        .collect()[0]
        .asDict()
    )

    for col_name, null_count in null_counts.items():
        if null_count and null_count > 0:
            pct = null_count / total * 100
            print(f"  {col_name:<22} {null_count:>5} nulls  ({pct:.1f}%)")

    print()


null_audit("bronze_drones")
null_audit("bronze_deliveries")
null_audit("bronze_flight_logs")

print("Null audit complete. These will be addressed in the Silver layer.")

── bronze_drones (100 rows) ─────────────────────────

── bronze_deliveries (5,000 rows) ─────────────────────────
  distance_km               50 nulls  (1.0%)

── bronze_flight_logs (30,000 rows) ─────────────────────────
  battery_level            875 nulls  (2.9%)
  gps_signal               609 nulls  (2.0%)
  weather_condition        568 nulls  (1.9%)

Null audit complete. These will be addressed in the Silver layer.
